In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/hc3_all.csv", header=0)

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (48644, 5)
Dataset NaN count:
id                     0
question               0
human_answers          0
chatgpt_answers        0
source             24322
dtype: int64


,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,reddit_eli5
4,4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,reddit_eli5


## Find missing values

In [17]:
# Are there missing values?
print(df.isnull().values.any())
print(df.isnull().head())

nan_count = np.sum(df.isnull(), axis = 0)
print(f"Dataset NaN count:\n{nan_count}")


missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

missing_summary

True
      id  question  human_answers  chatgpt_answers  source
0  False     False          False            False   False
1  False     False          False            False   False
2  False     False          False            False   False
3  False     False          False            False   False
4  False     False          False            False   False
Dataset NaN count:
id                     0
question               0
human_answers          0
chatgpt_answers        0
source             24322
dtype: int64


,missing_count,missing_percent
id,0,0.0
question,0,0.0
human_answers,0,0.0
chatgpt_answers,0,0.0
source,24322,50.0


We actually shouldn't need to correct any missing data in the source column as the data in that column are strings, and it doesn't include particularly important information for this project.

## Find duplicates

In [3]:
duplicate_count = df.duplicated().sum()

print(f"Duplicate rows: {duplicate_count}")

df[df.duplicated(keep=False)].sort_values(
    by=df.columns.tolist()
)

Duplicate rows: 0


,id,question,human_answers,chatgpt_answers,source


## Find inconsistencies

In [25]:
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    series = df[col].dropna().astype(str)

    whitespace_issues = series[
        series.str.strip() != series
    ]

    multiple_space_issues = series[
        series.str.contains(r"\s{2,}", regex=True)
    ]

    print(f"\n{col} Column:")
    print(f"Leading/trailing whitespace: {len(whitespace_issues)}")
    print(f"Multiple spaces: {len(multiple_space_issues)}")


question Column:
Leading/trailing whitespace: 3888
Multiple spaces: 426

human_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 39872

chatgpt_answers Column:
Leading/trailing whitespace: 0
Multiple spaces: 3686

source Column:
Leading/trailing whitespace: 0
Multiple spaces: 0


## Create the cleaned dataset

In [26]:
clean = df.copy()


# Clean text
text_columns = clean.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    clean[col] = (
        clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


clean.head()

,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,reddit_eli5
4,4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,reddit_eli5


## Saving the dataset

In [27]:
output_path = "../data/processed/cleaned_dataset.csv"

clean.to_csv(output_path, index=False)

print(f"Saved cleaned dataset to: {output_path}")
print(f"Original shape: {df.shape}")
print(f"Cleaned shape: {clean.shape}")

Saved cleaned dataset to: ../data/processed/cleaned_dataset.csv
Original shape: (48644, 5)
Cleaned shape: (48644, 5)
